# Critical Earth: Explainer Notebook

**DTU 02806 Social Data Analysis and Visualization**

This notebook is the technical companion to the narrative website. It documents the processed datasets, analysis choices, narrative genre, and visualization decisions behind the first-draft website.

In [ ]:
from pathlib import Path
import json

import geopandas as gpd
import pandas as pd


def find_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "data" / "processed").exists():
            return candidate
    raise FileNotFoundError("Could not locate project root from current working directory.")


ROOT = find_root(Path.cwd().resolve())
PROCESSED = ROOT / "data" / "processed"
WEBSITE = ROOT / "website"

economic = pd.read_csv(PROCESSED / "master_economic_timeseries.csv")
trade = pd.read_csv(PROCESSED / "master_supply_chain_trade.csv")
deposits = gpd.read_file(PROCESSED / "master_geo_deposits.geojson")
story_context = json.loads((WEBSITE / "data" / "story_context.json").read_text(encoding="utf-8"))
manifest = json.loads((WEBSITE / "visualizations" / "manifest.json").read_text(encoding="utf-8"))

target_minerals = [
    "Copper",
    "Lithium",
    "Graphite",
    "Neodymium",
    "Dysprosium",
    "Cobalt",
    "Gallium",
    "Platinum",
]


## 1. Motivation

**What is your dataset?**

The project uses three processed master datasets that summarize the full pipeline without exposing the website to raw-data dependencies:

- `data/processed/master_geo_deposits.geojson`: global deposit geography for the eight target minerals.
- `data/processed/master_economic_timeseries.csv`: production, prices, price proxies, and end-use shares.
- `data/processed/master_supply_chain_trade.csv`: stage shares, HHI concentration values, and normalized Comtrade trade-flow rows.

**Why did you choose this/these particular dataset(s)?**

I chose them because the research question cannot be answered with one data source. Geology explains where supply could come from, economic time series explain why these minerals matter, and supply-chain plus trade data explain where power actually sits. Together, the three masters let the analysis move from availability to dependence to trade re-routing.

**What was your goal for the end user's experience?**

The goal was a non-technical, self-contained experience for a reader who has not taken the course. The website should let that reader understand three things quickly:

1. These minerals matter for everyday technologies and the energy transition.
2. Geological availability is broader than the industrial chokepoints.
3. The world appears to be moving toward a partial split in trade, but not a fully bipolar system yet.

## 2. Basic Stats

**Write about your choices in data cleaning and preprocessing.**

The preprocessing decisions were intentionally conservative:

- Raw files were kept immutable in `data/raw/`, and all downstream work was regenerated into `data/processed/`.
- The project was reduced to eight focal minerals so the narrative stayed coherent.
- Multiple source families were normalized into three master products so the website and notebook could share one processed-data contract.
- Prices were rebased into an index-friendly form, mining and processing shares were standardized to percentages, and weights and currencies were kept in metric tonnes and USD.
- Comtrade rows were normalized into one supply-chain master so the website never had to query raw bilateral files directly.

**Write a short section that discusses the dataset stats, containing key points/plots from your exploratory data analysis.**

The key exploratory result is that the dataset combines broad geology with much tighter industrial concentration. Deposit geography spans many countries, but the latest stage-share summaries show very high concentration in processing. That gap between distributed deposits and concentrated downstream capability is the core empirical tension of the project.

In [ ]:
dataset_overview = pd.DataFrame(
    [
        {
            "dataset": "master_geo_deposits.geojson",
            "rows": len(deposits),
            "columns": len(deposits.columns),
            "note": f"{deposits['country'].nunique()} countries represented",
        },
        {
            "dataset": "master_economic_timeseries.csv",
            "rows": len(economic),
            "columns": len(economic.columns),
            "note": f"{economic['metric'].nunique()} metrics across the economic master",
        },
        {
            "dataset": "master_supply_chain_trade.csv",
            "rows": len(trade),
            "columns": len(trade.columns),
            "note": f"{trade['record_type'].nunique()} record types in the supply-chain master",
        },
    ]
)

latest_price = story_context['highlights']['price']
concentration = pd.DataFrame.from_dict(story_context['highlights']['concentration_2023'], orient='index')
end_use_summary = pd.DataFrame(story_context['highlights']['end_uses'])

display(dataset_overview)
display(concentration)
display(end_use_summary[['mineral', 'category', 'value', 'year']])
latest_price


## 3. Data Analysis

**Describe your data analysis and explain what you've learned about the dataset.**

The analysis combines four simple but defensible steps:

1. Compare deposit geography with later-stage concentration to separate geological possibility from industrial reality.
2. Use stage-share rows and HHI values to compare mining concentration with processing concentration.
3. Compare China's export destination mix in 2015 and 2023 to evaluate whether trade is splitting into blocs.
4. Use price and end-use context to explain why these supply-chain chokepoints matter economically and politically.

What I learned is that concentration increases downstream. In the 2023 stage-share view, processing is highly concentrated for all eight minerals, while mining is already highly concentrated for seven of the eight. The trade evidence does not support a clean bipolar split yet. Instead, it supports a partial rebalancing inside a still-concentrated global system.

**If relevant, talk about your machine-learning.**

No machine-learning model was used in the first draft. That was deliberate. The strongest findings came from transparent descriptive analysis: normalized price series, stage-share comparisons, HHI concentration, and trade-flow comparisons. For this research question, interpretability mattered more than model complexity.

In [ ]:
leaders = pd.DataFrame(story_context['highlights']['leaders_2023']).T.reset_index(drop=True)
bloc_share = pd.DataFrame(story_context['highlights']['china_bloc_share']).T.reset_index(names='year')

display(leaders[['mineral', 'mining_country', 'mining_share', 'processing_country', 'processing_share']])
display(bloc_share[['year', 'EU-27', 'United States', 'Other countries', 'western_bloc']])


## 4. Genre

**Which genre of data story did you use?**

The website uses a **Martini Glass** structure. It begins with a strongly author-driven sequence that walks the reader through stakes, geology, two supply-chain cases, trade evidence, and conclusion. It then gives the reader more freedom inside the interactive charts and the deposit map.

**Which tools did you use from each of the 3 categories of Visual Narrative (Figure 7 in Segel and Heer)? Why?**

- **Visual structuring:** repeated full-width section blocks, consistent dark-first styling, numbered sections, and a fixed country/mineral color system. These choices help the reader understand that every chart belongs to one coherent story world.
- **Highlighting:** section fact cards, paired before/after trade charts, and repeated emphasis on mining versus processing leaders. These tools point the reader toward the intended comparison without requiring technical explanation.
- **Transition guidance:** short narrative intros before each chart, anchor navigation, and a section order that moves from broad stakes to specific evidence. These choices reduce the cognitive jump between visualizations.

**Which tools did you use from each of the 3 categories of Narrative Structure (Figure 7 in Segel and Heer)? Why?**

- **Ordering:** the page is strictly linear from Section 0 to Section 6. This keeps the core argument easy to follow for a non-specialist reader.
- **Messaging:** the story uses declarative section titles, callout summaries, and a final takeaway grid so the main claim remains explicit.
- **Interactivity:** interactivity is limited but meaningful. The deposit map supports exploration, and the Plotly/Folium exports preserve hover or zoom where it helps comprehension. The point was guided exploration, not a dashboard.

## 5. Visualizations

**Explain the visualizations you've chosen. Why are they right for the story you want to tell?**

- **Mineral overview treemap:** a compact opener that introduces the eight-mineral scope without forcing the reader into tables.
- **Price index:** the clearest early signal of stress and volatility; useful as a narrative hook.
- **End-use treemap:** shows why each mineral matters in concrete terms such as batteries, catalysts, chips, and engines.
- **Deposit map:** the best way to show that geology is globally distributed and to provide the main reader-driven moment.
- **Mining vs processing slope chart:** the strongest analytical chart in the project because it makes the bottleneck visible immediately.
- **Production time series:** shows that rising output does not automatically resolve dependence.
- **Cobalt Sankey:** makes downstream concentration intuitive by following one material through a narrow path.
- **Trade flow comparison for 2015 and 2023:** directly addresses the research question with a before/after comparison.
- **China export timeline:** complements the two snapshots by showing whether the destination mix shifts over time.
- **HHI heatmap:** the cleanest final synthesis because it compresses the whole project into one concentration view.

The chosen set works because each section answers a different part of the same argument. The charts are not interchangeable. Each one is attached to a specific narrative job.

## 6. Discussion

**What went well?**

The processed-data architecture worked well. Reducing the project to three master datasets made the website, visualization exports, and notebook consistent. The mining-versus-processing contrast also turned out to be a strong explanatory device because it connects geology, industry, and geopolitics in one move.

**What is still missing? What could be improved? Why?**

The first draft still needs tighter copy editing, more polished mobile testing, and final contributor attribution. The trade argument would also benefit from a cleaner bloc definition and a more explicit sensitivity discussion around HS codes. Those improvements matter because the course deliverable should be persuasive for general readers but still methodologically transparent for the notebook audience.

**References and standards**

The project is built from processed versions of the source families documented in `AGENTS.md`, `project_plan_critical_earth_v3.md`, and `data/processed/README.md`: USGS critical-mineral geography, USGS historical statistics and MCS extracts, Our World in Data mine production, and UN Comtrade trade flows. The website only consumes processed datasets and standalone HTML exports, while this notebook documents the reasoning behind those transformations.

## 7. Contributions

**Who did what?**

Current draft attribution based on the local repo metadata:

- **ARHH:** led processed-data integration, Comtrade collection, visualization export pipeline, website implementation, and the first draft of this explainer notebook.

If this is a multi-person group submission, replace this draft attribution with a final named breakdown before hand-in. The course requirement explicitly asks for named responsibilities rather than a statement that everyone contributed equally.